In [1]:
# clone repo

import os

PROJECT_ROOT = "/content/Project_Generative_AI_for_Data_Augmentation"

if not os.path.exists(PROJECT_ROOT):
    !git clone https://github.com/Jorj91/Project_Generative_AI_for_Data_Augmentation.git {PROJECT_ROOT}

%cd {PROJECT_ROOT}

Cloning into '/content/Project_Generative_AI_for_Data_Augmentation'...
remote: Enumerating objects: 193, done.
remote: Counting objects: 100% (193/193), done.
remote: Compressing objects: 100% (163/163), done.
remote: Total 193 (delta 98), reused 76 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (193/193), 7.52 MiB | 5.58 MiB/s, done.
Resolving deltas: 100% (98/98), done.
/content/Project_Generative_AI_for_Data_Augmentation


In [2]:
# Dependency install
INSTALL_DEPS = True

if INSTALL_DEPS:
    !pip install -r requirements.txt -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.7 MB/s eta 0:00:00


In [5]:
import sys
sys.path.append(os.path.join(PROJECT_ROOT, "src"))

import importlib
import captioning
importlib.reload(captioning)
from captioning import run_captioning

In [6]:
# =============================
# CONTROLLED VERBOSITY
# =============================

# Disable HF download progress bars BEFORE importing anything HF-related

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

In [7]:
# Setup

import torch
import logging
from torchvision.datasets import OxfordIIITPet
import numpy as np
from torch.utils.data import Subset
from sklearn.model_selection import train_test_split
from transformers import Blip2Processor, Blip2ForConditionalGeneration
import random
from transformers.utils import logging as transformers_logging
from huggingface_hub.utils import logging as hf_logging


# Silence transformers & HF logs (keep only errors)
transformers_logging.set_verbosity_error()
hf_logging.set_verbosity_error()

logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

In [9]:
# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [10]:
PROJECT_ROOT # it must be /content/Project_Generative_AI_for_Data_Augmentation

'/content/Project_Generative_AI_for_Data_Augmentation'

In [11]:
# control flags
RUN_CAPTIONING = True
RUN_TEXT_VARIATION = False
RUN_IMAGE_GENERATION = False
RUN_TRAINING = False

In [12]:
# dataset loading

dataset_train = OxfordIIITPet(
    root=os.path.join(PROJECT_ROOT, "data", "raw"),
    split="trainval",
    download=True
)

dataset_test = OxfordIIITPet(
    root=os.path.join(PROJECT_ROOT, "data", "raw"),
    split="test",
    download=True
)

print("Train size:", len(dataset_train))
print("Test size:", len(dataset_test))

100%|██████████| 792M/792M [00:50<00:00, 15.8MB/s]
100%|██████████| 19.2M/19.2M [00:01<00:00, 9.89MB/s]


Train size: 3680
Test size: 3669


In [13]:
# extract labels
labels = dataset_train._labels
indices = np.arange(len(dataset_train))

# perform stratified split
train_small_idx, _ = train_test_split(
    indices,
    train_size=0.30,
    stratify=labels,
    random_state=42
)

SPLIT_DIR = os.path.join(PROJECT_ROOT, "data", "splits")
os.makedirs(SPLIT_DIR, exist_ok=True)

np.save(os.path.join(SPLIT_DIR, "train_small_indices.npy"), train_small_idx)

In [19]:
# # run for ALL
# # create subset dataset from training set
dataset_train_small = Subset(dataset_train, train_small_idx)

In [23]:
# run for 10
dataset_train_small_10 = Subset(dataset_train, train_small_idx[:10])

In [24]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [25]:
processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")

model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    torch_dtype=torch.float16 # to reduce GPU memory usage
)

model.to(device)
model.eval()

Blip2ForConditionalGeneration(
  (vision_model): Blip2VisionModel(
    (embeddings): Blip2VisionEmbeddings(
      (patch_embedding): Conv2d(3, 1408, kernel_size=(14, 14), stride=(14, 14))
    )
    (encoder): Blip2Encoder(
      (layers): ModuleList(
        (0-38): 39 x Blip2EncoderLayer(
          (self_attn): Blip2Attention(
            (qkv): Linear(in_features=1408, out_features=4224, bias=True)
            (projection): Linear(in_features=1408, out_features=1408, bias=True)
          )
          (layer_norm1): LayerNorm((1408,), eps=1e-06, elementwise_affine=True)
          (mlp): Blip2MLP(
            (activation_fn): GELUActivation()
            (fc1): Linear(in_features=1408, out_features=6144, bias=True)
            (fc2): Linear(in_features=6144, out_features=1408, bias=True)
          )
          (layer_norm2): LayerNorm((1408,), eps=1e-06, elementwise_affine=True)
        )
      )
    )
    (post_layernorm): LayerNorm((1408,), eps=1e-06, elementwise_affine=True)
  )
  (qf

In [26]:
CAPTION_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "captions",
    "captions_train_small_10.json"
)

if RUN_CAPTIONING:
    captions_dict = run_captioning(
        # dataset_train_small=dataset_train_small,
        dataset_train_small=dataset_train_small_10,
        model=model,
        processor=processor,
        device=device,
        output_path=CAPTION_PATH
    )

100%|██████████| 10/10 [00:08<00:00,  1.13it/s]

Full caption generation completed.


In [ ]:
# import nbformat
# import os

# def hard_clean_notebook(path):
#     nb = nbformat.read(path, as_version=4)

#     if "widgets" in nb.metadata:
#         del nb.metadata["widgets"]

#     for cell in nb.cells:
#         if "widgets" in cell.get("metadata", {}):
#             del cell["metadata"]["widgets"]

#     nbformat.write(nb, path)
#     print(f"Cleaned: {os.path.basename(path)}")


# # 🔥 Walk entire project and clean every notebook
# for root, _, files in os.walk(PROJECT_ROOT):
#     for file in files:
#         if file.endswith(".ipynb"):
#             hard_clean_notebook(os.path.join(root, file))

# print("All notebooks in project HARD cleaned.")